# train-eval-mode-branch — worked example 3: Temporarily eval a model and restore train mode after

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `train-eval-mode-branch`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Sometimes you need to run a quick inference pass in the middle of a training loop without permanently switching the model to eval mode. The cleanest approach is to snapshot `model.training`, switch to eval, do the inference, then restore the original mode. This ensures your training loop is not accidentally broken if the inference call is inside a subroutine.

## Worked solution

**Step 1 – Snapshot the current mode.** `was_training = model.training` captures whether the model was in train mode before we touch it.

**Step 2 – Switch to eval and run inference.** We call `model.eval()` and run a forward pass. Under `t.no_grad()` we also avoid building a computation graph, which saves memory.

**Step 3 – Restore original mode.** `model.train(was_training)` sets the flag back to whatever it was. If the caller had `model.train()` active, calling this restores train mode; if it was already eval, we stay eval.

**Step 4 – Verify both outputs.** The function returns both the mid-loop inference output and a flag showing the mode was restored. A Dropout-containing model will produce stochastic outputs AFTER restoration if it was in train mode before, confirming the restore worked.

In [ ]:
import torch as t
import torch.nn as nn

def worked3_eval_and_restore(model, x_inference, x_train):
    """
    Run an inference pass in eval mode, then restore original mode.
    Returns dict with inference output, restored-mode output, and flags.
    """
    # Snapshot current mode
    was_training = model.training

    # Eval pass (no grad, deterministic)
    model.eval()
    with t.no_grad():
        out_eval = model(x_inference)
    mode_during_eval = model.training  # False

    # Restore original mode
    model.train(was_training)
    mode_after_restore = model.training  # same as was_training

    # Forward in restored mode (will be stochastic if was train)
    out_restored = model(x_train)

    return {
        'out_eval': out_eval,
        'out_restored': out_restored,
        'was_training': was_training,
        'mode_during_eval': mode_during_eval,
        'mode_after_restore': mode_after_restore,
        'correctly_restored': mode_after_restore == was_training,
    }

# Demo
t.manual_seed(55)
model = nn.Sequential(nn.Linear(4, 4), nn.Dropout(0.5), nn.Linear(4, 2))
model.train()  # start in train mode
x1 = t.randn(3, 4)
x2 = t.randn(3, 4)
result = worked3_eval_and_restore(model, x1, x2)
print('was_training:', result['was_training'])
print('mode_during_eval:', result['mode_during_eval'])
print('mode_after_restore:', result['mode_after_restore'])
print('correctly_restored:', result['correctly_restored'])